In [ ]:
import csv
from contextlib import ExitStack
from pathlib import Path

INPUT = Path("/kaggle/input")
OUTPUT = Path("/kaggle/working/shards")
OUTPUT.mkdir(parents=True, exist_ok=True)

NUM_SHARDS = 30
ESTIMATED_FILES_PER_HOUR = 112_140

source_script = next(INPUT.rglob("src/batch/download_data.py"))
source_project = source_script.parents[2]  # src/batch/download_data.py -> project root
manifest = source_project / "data/metadata/archive_manifest.csv"

# Có downloaded_local.csv thì loại file đã tải; không có thì bỏ qua.
completed = {}

for path in INPUT.rglob("downloaded_local.csv"):
    print("Đọc:", path, flush=True)
    with path.open(newline="", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            completed[row["key"]] = int(row["size"])

counts = [0] * NUM_SHARDS
sizes = [0] * NUM_SHARDS
scanned = skipped = assigned = 0

with ExitStack() as stack:
    writers = []

    for shard_id in range(1, NUM_SHARDS + 1):
        stream = stack.enter_context(
            (OUTPUT / f"shard_{shard_id:05d}.csv").open(
                "w", newline="", encoding="utf-8"
            )
        )
        writer = csv.writer(stream)
        writer.writerow(["key", "size"])
        writers.append(writer)

    source = stack.enter_context(
        manifest.open(newline="", encoding="utf-8-sig")
    )

    for row in csv.DictReader(source):
        scanned += 1
        key, size = row["key"], int(row["size"])

        if completed.get(key) == size:
            skipped += 1
        else:
            shard_index = assigned % NUM_SHARDS
            writers[shard_index].writerow([key, size])
            counts[shard_index] += 1
            sizes[shard_index] += size
            assigned += 1

        if scanned % 1_000_000 == 0:
            print(
                f"Đã đọc {scanned:,} | "
                f"Bỏ qua {skipped:,} | Đã chia {assigned:,}",
                flush=True,
            )

with (OUTPUT / "shard_index.csv").open(
    "w", newline="", encoding="utf-8"
) as f:
    writer = csv.writer(f)
    writer.writerow([
        "shard_id", "filename", "files", "bytes", "estimated_hours"
    ])

    for index in range(NUM_SHARDS):
        shard_id = index + 1
        hours = counts[index] / ESTIMATED_FILES_PER_HOUR

        writer.writerow([
            shard_id,
            f"shard_{shard_id:05d}.csv",
            counts[index],
            sizes[index],
            round(hours, 2),
        ])

        print(
            f"Shard {shard_id:02d}: {counts[index]:,} file | "
            f"{sizes[index] / 1e9:.3f} GB | "
            f"dự kiến {hours:.2f} giờ"
        )

        if sizes[index] > 15_000_000_000:
            print("  Lưu ý: shard này vượt mức dự phòng 15 GB.")

print(f"\nĐã chia {assigned:,} file thành {NUM_SHARDS} shard.")
print(f"Đã bỏ qua {skipped:,} file có sẵn.")
print(f"Output: {OUTPUT}")